In [1]:
!pip freeze | grep scikit-learn
!python -V


scikit-learn==1.6.0
Python 3.12.8


In [1]:
import pickle
import pandas as pd

with open('model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)


c:\Program Files\Python312\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.5.0 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Program Files\Python312\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.5.0 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [2]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

# df = read_data('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_????-??.parquet')
df = read_data('../../../../data/yellow_tripdata_2023-03.parquet')


In [ ]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)
print(y_pred.std()) # q1


6.247488852238703


In [4]:
f'{df.loc[5654]["tpep_pickup_datetime"].month:02d}'


'03'

In [ ]:
dfidx = df.index.astype("str")
df["ride_id"] = (
    df["tpep_pickup_datetime"].map(lambda row: f"{row.year:04d}/{row.month:02d}_")
    + dfidx
)


In [ ]:
df_result = df.loc[:][['ride_id','duration']]


In [ ]:
df_result.to_parquet(
    'df_result_output_file.parquet',
    engine='pyarrow',
    compression=None,
    index=False
)
